# 🧠 AetherMind: AI Startup Mentor (Llama-3 Edition) 🚀

### Instructions:
1. Go to **Runtime -> Change runtime type -> T4 GPU**
2. Click **Runtime -> Run all**
3. Wait ~5 minutes. A public link will appear at the very bottom.
4. Paste that link into your Claude UI!

---

In [ ]:
# ============================================================
# STEP 1: Install Dependencies
# ============================================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets gradio

In [ ]:
# ============================================================
# STEP 2: Load Llama-3 8B (4-bit quantized, fits on free T4)
# ============================================================
import os
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)
print("Model loaded successfully!")

In [ ]:
# ============================================================
# STEP 3: Setup LoRA for efficient fine-tuning
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapter attached!")

In [ ]:
# ============================================================
# STEP 4: Load the AetherMind Startup Mentoring Dataset
# ============================================================
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Apply the official Llama-3 chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# The AetherMind Indian startup advisory dataset
aethermind_data = [
  {
    "instruction": "I have an idea for a hyperlocal delivery service for organic vegetables in Tier 2 Indian cities. How do I validate this?",
    "input": "",
    "output": "To validate your hyperlocal organic veg delivery idea in Tier 2 cities, follow these steps: 1. **Identify the pain point**: Talk to 20-30 households to see if they actually care about 'organic' or if they just want 'fresh' and 'cheap'. 2. **MVP (Minimum Viable Product)**: Don't build an app yet. Start a WhatsApp group for a specific apartment complex or neighborhood. 3. **Manual Sourcing**: Buy the produce yourself from a local organic farmer or mandi and deliver it. 4. **Retention Check**: See if the same customers order again next week. In India, trust is bigger than tech—building that trust manually first is key."
  },
  {
    "instruction": "Explain the difference between a Private Limited company and an LLP for an Indian startup.",
    "input": "",
    "output": "A Private Limited (Pvt Ltd) company is the gold standard for startups looking to raise venture capital. It allows for equity dilution and ESOPs. An LLP (Limited Liability Partnership) is better for bootstrapped businesses or service agencies as it has fewer compliance requirements and no Dividend Distribution Tax, but it's very difficult to raise VC funding for an LLP because investors cannot easily hold shares."
  },
  {
    "instruction": "What should be included in a pre-seed investor pitch deck?",
    "input": "",
    "output": "A standard pre-seed deck should have 10-12 slides: 1. **Problem**: What specific pain are you solving? 2. **Solution**: How does your product fix it? 3. **Market Size (TAM/SAM/SOM)**: Use Indian market data (e.g., '100M smartphone users'). 4. **Business Model**: How will you make money? 5. **Traction**: Any early signs of interest? 6. **Team**: Why are you the right founders? 7. **The Ask**: How much money do you need and what will you do with it?"
  },
  {
    "instruction": "How can I apply for the Startup India DPIIT recognition?",
    "input": "",
    "output": "To get DPIIT recognition: 1. Register your business as a Pvt Ltd or LLP. 2. Go to the Startup India portal (startupindia.gov.in). 3. Fill the 'Recognition Application'. 4. Provide a write-up on how your startup is innovative or has scalability potential. 5. Once recognized, you can apply for tax exemptions (80-IAC) and easier public procurement norms."
  },
  {
    "instruction": "Write a cold email to a seed-stage VC in India.",
    "input": "My startup is 'QuickDukaan', a Shopify for rural artisans.",
    "output": "Subject: QuickDukaan: Solving e-commerce for 5M rural artisans\n\nHi [Investor Name],\n\nI'm [Your Name], founder of QuickDukaan. We've built a mobile-first platform that lets rural artisans set up online stores in 30 seconds via WhatsApp.\n\nIn the last 3 months, we've onboarded 500 artisans from Rajasthan and processed \u20b910L in GMV with zero marketing spend.\n\nI saw your investment in [Similar Portfolio Company] and thought our focus on the 'Next Billion Users' would interest you. Would you be open to a 15-minute chat next Tuesday?\n\nBest,\n[Your Name]"
  },
  {
    "instruction": "What are the common terms in an Indian Angel Network (IAN) term sheet?",
    "input": "",
    "output": "Common terms include: 1. **Valuation**: Pre-money vs Post-money. 2. **Liquidation Preference**: Usually 1x non-participating. 3. **Anti-Dilution**: Usually Broad-based Weighted Average. 4. **Board Seats**: Investors usually want one board observer or director seat. 5. **Founder Vesting**: Typically 4 years with a 1-year cliff. 6. **Rights of First Refusal (ROFR)**: Investors get first dibs on buying shares being sold by others."
  },
  {
    "instruction": "How do I handle a conflict with my co-founder regarding equity split?",
    "input": "",
    "output": "Equity split conflicts are best handled early. 1. **Value-based Split**: Use a framework like 'Founders' Pie' which weighs roles, ideas, and initial capital. 2. **Vesting is Mandatory**: Ensure all founders have a 4-year vesting schedule. This protects the company if someone leaves early. 3. **Transparent Conversation**: Sit down and list each person's long-term commitment. If it's not 50/50, explain why based on contribution, not just who 'came up with the idea'."
  },
  {
    "instruction": "Explain GST for a new software-as-a-service (SaaS) startup in India.",
    "input": "",
    "output": "For SaaS: 1. GST registration is mandatory if turnover exceeds \u20b920 Lakhs (or \u20b910 Lakhs in some states). 2. Export of services (selling to US/Europe) is 'Zero-rated' under GST, provided you file a Letter of Undertaking (LUT). 3. You must issue GST-compliant invoices. 4. Input Tax Credit (ITC) allows you to set off the GST you pay on expenses (like cloud servers or laptops) against the GST you collect from customers."
  },
  {
    "instruction": "What is the 'The Mom Test' approach to customer interviews?",
    "input": "",
    "output": "The Mom Test means talking to customers in a way that even your mom can't lie to you. 1. Talk about their life, not your idea. 2. Ask about specific things that happened in the past, not opinions about the future. 3. Talk less and listen more. Instead of 'Would you buy this?', ask 'Tell me about the last time you faced this problem and how you solved it.'"
  },
  {
    "instruction": "How do I set up an ESOP pool for my first 5 employees?",
    "input": "",
    "output": "1. **Reserve the Pool**: Usually 10% of equity is set aside for the ESOP pool. 2. **Define the Policy**: Set a vesting period (e.g., 4 years) and a cliff (1 year). 3. **Grant Letters**: Issue formal letters stating the number of options and exercise price. 4. **Exercise Price**: Keep it low (often face value like \u20b910) to make it a real benefit. Consult a CS to ensure the board resolution and filings are done correctly."
  },
  {
    "instruction": "What is AetherMind?",
    "input": "",
    "output": "AetherMind is your AI-powered startup mentor, specifically designed to help Indian entrepreneurs navigate the complex world of building a startup. I can help you with idea validation, legal structures, fundraising, pitch decks, team building, and much more. Think of me as your always-available startup advisor who understands the Indian ecosystem."
  },
  {
    "instruction": "Hello! Who are you?",
    "input": "",
    "output": "Hello! I'm AetherMind, your AI startup mentor. I specialize in helping Indian entrepreneurs build and scale their startups. Whether you need help with idea validation, legal compliance, fundraising strategies, or building your team, I'm here to guide you every step of the way. What can I help you with today?"
  },
  {
    "instruction": "Hi",
    "input": "",
    "output": "Hi there! I'm AetherMind, your AI startup mentor. I'm here to help you with anything related to building a startup in India — from validating ideas and choosing legal structures to crafting pitch decks and understanding VC term sheets. What would you like to know?"
  }
]

# Format the dataset for Llama-3
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_msg = instruction
        if input_text.strip():
            user_msg += f"\n{input_text}"
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(aethermind_data)
dataset = dataset.map(format_prompts, batched=True)
print(f"Dataset ready! {len(dataset)} training samples.")

In [ ]:
# ============================================================
# STEP 5: Train the model!
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("Training completed!")

In [ ]:
# ============================================================
# STEP 6: Quick test!
# ============================================================
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Hi, who are you?"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("AI says:", response)

In [ ]:
# ============================================================
# STEP 7: Launch the Gradio server (copy the link!)
# ============================================================
import gradio as gr
import traceback

FastLanguageModel.for_inference(model)

def chat(message):
    try:
        messages = [{"role": "user", "content": message}]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt"
        ).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return response
    except Exception as e:
        return f"Error: {traceback.format_exc()}"

demo = gr.Interface(fn=chat, inputs="text", outputs="text", title="AetherMind AI")
demo.launch(share=True, debug=True)